### Computing RDF2VEC for SWDF dataset

In [1]:
import io
import csv
import requests

res = requests.get(
    "http://127.0.0.1:8890/sparql/",
    params={
		"query": """
			SELECT DISTINCT ?id 
			FROM <swdf>
			WHERE { 
			{ ?id [] [] }
			UNION
			{ [] ?id [] }
			UNION
			{ [] [] ?id }
			}
		""",
		"format": "csv",
	},
)
str_io = io.StringIO(res.text)
reader = csv.DictReader(str_io)
entities = [row["id"] for row in reader]

In [5]:
from pyrdf2vec import RDF2VecTransformer
from pyrdf2vec.embedders import Word2Vec
from pyrdf2vec.graphs import KG
from pyrdf2vec.walkers import RandomWalker

In [6]:
# %%prun -s cumulative

# Define our knowledge graph (here: DBPedia SPARQL endpoint).
knowledge_graph = KG(
    location="http://127.0.0.1:8890/sparql/",
    skip_verify=True,
)

# Create our transformer, setting the embedding & walking strategy.
transformer = RDF2VecTransformer(
    embedder=Word2Vec(epochs=10),
    walkers=[RandomWalker(
        max_depth=4,
        max_walks=10,
        with_reverse=False,
        n_jobs=10,
    )],
    verbose=1
)

# Get our embeddings.
embeddings, literals = transformer.fit_transform(knowledge_graph, entities)

100%|██████████| 76881/76881 [05:45<00:00, 222.79it/s] 


Extracted 279619 walks for 76881 entities (345.2019s)
Fitted 279619 walks (7.4338s)


In [ ]:
# Backing up the results into pickle file
import pickle

with open("../datasets/swdf/rdf2vec100dim.pkl", "wb") as f:
    pickle.dump(
        dict(zip(entities, embeddings)),
        f
    )

### Computing occurrences for SWDF dataset

In [8]:
import io
import csv
import requests

res = requests.get(
    "http://127.0.0.1:8890/sparql/",
    params={
		"query": """
			SELECT DISTINCT ?id, COUNT(?id) AS ?count
			FROM <swdf>
			WHERE { 
				{ ?id [] [] }
				UNION
				{ [] ?id [] }
				UNION
				{ [] [] ?id }
			}
		""",
		"format": "csv",
	},
)
str_io = io.StringIO(res.text)
reader = csv.DictReader(str_io)
count_vals = {row["id"]: int(row["count"]) for row in reader}

In [ ]:
import pickle

with open("../datasets/swdf/counts.pkl", "wb") as f:
	pickle.dump(count_vals, f)